## Data Prep

**INSTALL DARTS DI COLAB**

In [97]:
!pip install darts
!pip install xgboost catboost -q
!pip install lightgbm xgboost catboost optuna -q
!pip install optuna-integration[lightgbm]

**IMPORT LIBRARY YANG DIBUTUHKAN**

In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from darts import TimeSeries
from darts.models import AutoARIMA
from darts.metrics import smape
from darts.models import RNNModel
import random
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import optuna

**BUAT SEED AGAR HASIL TETAP, SEPERTI CHECKPOINT**

In [99]:
seed = 404
random.seed(seed)
np.random.seed(seed)

**MEMUAT DATA DARI GITHUB PRIBADI**

In [100]:
# URL Data di Github
d1 = 'https://raw.githubusercontent.com/Rashifalmas/Pedas2025/refs/heads/main/dataCarSale2021-2025.csv'
d2 = 'https://raw.githubusercontent.com/Rashifalmas/Pedas2025/refs/heads/main/Tingkat%20Inflasi%20Harga%20Konsumen%20Nasional%20Tahun%20Kalender%20(Y-to-D)%20%20(2022%3D100).csv'
d3 = 'https://github.com/Rashifalmas/Pedas2025/blob/main/BI-7Day-RR.xlsx?raw=true'

dfMain = pd.read_csv(d1)
dfBI = pd.read_excel(d3, skiprows=4, usecols=[1, 2])
dfIHK = pd.read_csv(d2, skiprows=2, nrows=12)

**DATA UTAMA**

In [101]:
dfMain.head()

,waktu,DAIHATSU,HONDA,MITSUBISHI,SUZUKI,TOYOTA
0,2021-01-01,8993.0,7231.0,9108.0,6400.0,15474.0
1,2021-02-01,9412.0,6812.0,7567.0,4600.0,14645.0
2,2021-03-01,16770.0,11350.0,13088.0,8669.0,26034.0
3,2021-04-01,15861.0,8474.0,13820.0,8100.0,22618.0
4,2021-05-01,8310.0,5832.0,10160.0,5341.0,17361.0


**mengubah tipe data untuk waktu**

In [102]:
# Ubah tipe data kolom waktu pada Data Utama
dfMain['waktu'] = pd.to_datetime(dfMain['waktu'], format='%Y-%m-%d')
dfMain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   waktu       54 non-null     datetime64[ns]
 1   DAIHATSU    54 non-null     float64       
 2   HONDA       54 non-null     float64       
 3   MITSUBISHI  54 non-null     float64       
 4   SUZUKI      54 non-null     float64       
 5   TOYOTA      54 non-null     float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 2.7 KB


**DATA SUKU BUNGA**

In [103]:
# Data Suku Bunga
dfBI.head()

,Tanggal,BI-7Day-RR
0,16 Juli 2025,5.25 %
1,18 Juni 2025,5.50 %
2,21 Mei 2025,5.50 %
3,23 April 2025,5.75 %
4,19 Maret 2025,5.75 %


**mapping bulan**

In [104]:
# Mapping dari bulan Bahasa Indonesia menjadi angka agar bisa terdeteksi Pandas
bulan_map = {
    'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

for bulan_indo, bulan_angka in bulan_map.items():
    dfBI['Tanggal'] = dfBI['Tanggal'].str.replace(bulan_indo, bulan_angka)

**ubah tipe data nya juga**

In [105]:
# Ubah tipe data kedua kolom pada Data Suku Bunga
dfBI['Tanggal'] = pd.to_datetime(dfBI['Tanggal'], format='%d %m %Y')
dfBI['BI-7Day-RR'] = dfBI['BI-7Day-RR'].str.replace(' %', '').astype(float)
dfBI.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Tanggal     55 non-null     datetime64[ns]
 1   BI-7Day-RR  55 non-null     float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1012.0 bytes


**resample agar bisa merge ke data utama**

In [106]:
# Ubah format lalu resample agar bisa menyesuaikan Data Utama
dfBI = dfBI.rename(columns={'Tanggal': 'waktu'})
dfBI = dfBI.set_index('waktu')
dfBI = dfBI.resample('MS').last()
dfBI = dfBI.reset_index()
dfBI.head()

,waktu,BI-7Day-RR
0,2021-01-01,3.75
1,2021-02-01,3.50
2,2021-03-01,3.50
3,2021-04-01,3.50
4,2021-05-01,3.50


In [107]:
# Konversi ke Darts TimeSeries
seri_bi_rate = TimeSeries.from_dataframe(dfBI, 'waktu', 'BI-7Day-RR')

# Inisialisasi model AutoARIMA
model_arima_bi = AutoARIMA()

# backtesting untuk prediksi 1 langkah ke depan
historical_forecasts = model_arima_bi.historical_forecasts(
    seri_bi_rate,
    start=0.8,
    forecast_horizon=1, # Aturan 2: Prediksi 1 langkah ke depan
    verbose=True
)
smape_score = smape(seri_bi_rate, historical_forecasts)

print(f"\nEvaluasi Backtesting sMAPE untuk Suku Bunga BI: {smape_score:.2f}%")

historical forecasts: 100%|██████████| 12/12 [00:04<00:00,  2.60it/s]


Evaluasi Backtesting sMAPE untuk Suku Bunga BI: 2.21%


In [108]:
# Latih model pada seluruh data (Jan 2021 - Jul 2025)
model_arima_bi.fit(seri_bi_rate)

# Prediksi 1 langkah ke depan (Agustus 2025)
prediksi_bi_rate_agustus = model_arima_bi.predict(n=1)
nilai_prediksi = prediksi_bi_rate_agustus.first_value()

print(f"\nPrediksi Final Suku Bunga BI untuk Agustus 2025: {nilai_prediksi:.2f}%")


Prediksi Final Suku Bunga BI untuk Agustus 2025: 5.08%


**DATA INFLASI**

In [109]:
# Data Inflasi
dfIHK.head()

,Bulan,2006,2007,2008,2009,2010,2011,2012,2013,2014,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Januari,1.36,1.04,1.77,-0.07,0.84,0.89,0.76,1.03,1.07,...,0.51,0.97,0.62,0.32,0.39,0.26,0.56,0.34,0.04,-0.76
1,Februari,1.95,1.67,2.44,0.14,1.14,1.03,0.81,1.79,1.33,...,0.42,1.21,0.79,0.24,0.66,0.36,0.54,0.50,0.41,-1.24
2,Maret,1.98,1.91,3.41,0.36,0.99,0.70,0.88,2.43,1.41,...,0.62,1.19,0.99,0.35,0.76,0.44,1.20,0.68,0.93,0.39
3,April,2.03,1.74,4.01,0.05,1.15,0.39,1.09,2.32,1.39,...,0.16,1.28,1.09,0.80,0.84,0.58,2.15,1.01,1.19,1.56
4,Mei,2.41,1.84,5.47,0.10,1.44,0.51,1.15,2.30,1.56,...,0.40,1.67,1.30,1.48,0.90,0.90,2.56,1.10,1.16,1.19


**resampling agar sesuai data utama**

In [110]:
# Mengubah format dari 'lebar' ke 'panjang' menggunakan melt.
dfIHK = pd.melt(dfIHK,
                      id_vars=['Bulan'],
                      var_name='Tahun',
                      value_name='IHK')

# Buat kolom sementara untuk menggabungkan bulan dan tahun
dfIHK['waktu'] = dfIHK['Bulan'] + ' ' + dfIHK['Tahun']

# Ganti nama bulan dengan angka
for bulan_indo, bulan_angka in bulan_map.items():
    dfIHK['waktu'] = dfIHK['waktu'].str.replace(bulan_indo, bulan_angka)

# Ubah string tanggal menjadi format datetime (tanggal-bulan-tahun)
dfIHK['waktu'] = pd.to_datetime(dfIHK['waktu'], format='%m %Y')

# Ubah IHK menjadi tipe data angka (float), ganti koma dengan titik jika perlu
dfIHK['IHK'] = dfIHK['IHK'].astype(float)

# Pilih kolom yang kita butuhkan dan urutkan berdasarkan waktu
dfIHK = dfIHK[['waktu', 'IHK']].sort_values(by='waktu').reset_index(drop=True)

# Filter data dari Januari 2021 hingga Juli 2025
start_date = '2021-01-01'
end_date = '2025-07-01'
dfIHK = dfIHK[(dfIHK['waktu'] >= start_date) & (dfIHK['waktu'] <= end_date)].copy()


**GABUNG KETIGA DATA**

In [111]:
dfFinal = pd.merge(dfMain, dfBI, on='waktu', how='left')
dfFinal = pd.merge(dfFinal, dfIHK, on='waktu', how='left')

# Buat DataFrame baru untuk Agustus 2025
df_agustus = pd.DataFrame({'waktu': [pd.to_datetime('2025-08-01')]})

# Gabungkan data historis dengan baris Agustus.
dfFinal = pd.concat([dfFinal, df_agustus], ignore_index=True)

# Isi nilai BI Rate dengan hasil prediksi model ARIMA Anda
dfFinal.loc[dfFinal['waktu'] == '2025-08-01', 'BI-7Day-RR'] = nilai_prediksi

# Isi nilai IHK dengan metode forward fill
dfFinal['IHK'].fillna(method='ffill', inplace=True)

/tmp/ipython-input-137527521.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfFinal['IHK'].fillna(method='ffill', inplace=True)
/tmp/ipython-input-137527521.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dfFinal['IHK'].fillna(method='ffill', inplace=True)


In [112]:
# Set index waktu
df = dfFinal.set_index('waktu')

# Ambil hanya 2024-2025
dfFinal = df.loc['2024':'2025']


In [113]:
dfFinal = dfFinal.reset_index()
dfFinal.head()

,waktu,DAIHATSU,HONDA,MITSUBISHI,SUZUKI,TOYOTA,BI-7Day-RR,IHK
0,2024-01-01,7348.196,3222.119,5076.613,3242.0,10366.353,6.00,0.04
1,2024-02-01,6021.329,3829.318,4349.666,2999.0,10770.845,6.00,0.41
2,2024-03-01,6591.920,5212.549,4708.298,3503.0,12301.583,6.00,0.93
3,2024-04-01,7510.349,3435.000,4898.000,2555.0,9116.144,6.25,1.19
4,2024-05-01,9336.778,5055.000,5522.211,2736.0,9445.956,6.25,1.16


**FE**

In [114]:
# Asumsikan 'dfGabungan' adalah DataFrame Anda dari Jan 2021 - Agu 2025
# dengan NaN pada kolom merek di bulan Agustus

# --- Lakukan Rekayasa Fitur pada Seluruh DataFrame ---
df_fitur = dfFinal.copy()
kolom_untuk_fitur = ['TOYOTA', 'DAIHATSU', 'HONDA', 'MITSUBISHI', 'SUZUKI', 'BI-7Day-RR', 'IHK']

for kolom in kolom_untuk_fitur:
    df_fitur[f'{kolom}_rolling_mean_3'] = df_fitur[kolom].rolling(window=3).mean()
    for i in range(1, 4):
        df_fitur[f'{kolom}_lag_{i}'] = df_fitur[kolom].shift(i)

# --- Bersihkan ---
df_fitur_clean = df_fitur.dropna(subset=['TOYOTA_lag_3']) # Cukup cek salah satu lag terakhir

print("--- DataFrame dengan Fitur Lengkap (5 baris terakhir) ---")
print(df_fitur_clean.tail())


# --- Pisahkan Data untuk Latih dan Prediksi ---

# Data untuk melatih model
df_latih = df_fitur_clean.dropna(subset=['TOYOTA'])

# Data untuk diprediksi
df_prediksi = df_fitur_clean[df_fitur_clean['TOYOTA'].isnull()]

--- DataFrame dengan Fitur Lengkap (5 baris terakhir) ---
        waktu  DAIHATSU     HONDA  MITSUBISHI    SUZUKI     TOYOTA  \
15 2025-04-01  6075.811  1590.411      5197.0  2767.379  13779.300   
16 2025-05-01  7537.632  3148.000      5897.0  2004.813   9313.266   
17 2025-06-01  5992.367  2360.797      6501.0  3499.336   8712.383   
18 2025-07-01  6354.101  4171.007      6121.0  2666.235   7617.122   
19 2025-08-01       NaN       NaN         NaN       NaN        NaN   

    BI-7Day-RR   IHK  TOYOTA_rolling_mean_3  TOYOTA_lag_1  ...  SUZUKI_lag_2  \
15    5.750000  1.56           14572.084333     13711.773  ...      3047.704   
16    5.500000  1.19           12268.113000     13779.300  ...      3223.220   
17    5.500000  1.38           10601.649667      9313.266  ...      2767.379   
18    5.250000  1.69            8547.590333      8712.383  ...      2004.813   
19    5.083333  1.69                    NaN      7617.122  ...      3499.336   

    SUZUKI_lag_3  BI-7Day-RR_rolling_mea

In [116]:
df_gabungan = pd.concat([df_latih, df_prediksi], ignore_index=True)

# Membuat Fitur Waktu & Fourier
df_gabungan['bulan'] = df_gabungan['waktu'].dt.month
df_gabungan['sin_bulan'] = np.sin(2 * np.pi * df_gabungan['bulan'] / 12)
df_gabungan['cos_bulan'] = np.cos(2 * np.pi * df_gabungan['bulan'] / 12)

# Membuat Fitur Rolling & Lag
kolom_untuk_fitur = ['TOYOTA', 'DAIHATSU', 'HONDA', 'MITSUBISHI', 'SUZUKI', 'BI-7Day-RR', 'IHK']
for kolom in kolom_untuk_fitur:
    for window in [3, 6]:
        df_gabungan[f'{kolom}_rolling_mean_{window}'] = df_gabungan[kolom].rolling(window=window).mean()
    for i in [1, 2, 3, 6]:
        df_gabungan[f'{kolom}_lag_{i}'] = df_gabungan[kolom].shift(i)

# bersihkan NaN
df_fitur = df_gabungan.dropna(subset=['TOYOTA_lag_6']).reset_index(drop=True)
df_latih_baru = df_fitur[df_fitur['waktu'] <= '2025-07-01']
df_prediksi_baru = df_fitur[df_fitur['waktu'] > '2025-07-01']

print("Rekayasa fitur lanjutan selesai.")

Rekayasa fitur lanjutan selesai.


**MODELLING**

In [117]:
def smape(y_true, y_pred):
    numerator = 2 * np.abs(y_pred - y_true)
    denominator = np.abs(y_true) + np.abs(y_pred)
    ratio = np.where(denominator == 0, 0, numerator / denominator)
    return np.mean(ratio) * 100

def objective(trial, X, y):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 404,
        'n_jobs': -1
    }
    model = LGBMRegressor(**params)

    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  callbacks=[optuna.integration.LightGBMPruningCallback(trial, "l1")])
        preds = model.predict(X_test)
        scores.append(smape(y_test, preds))

    return np.mean(scores)

# --- PROSES PEMODELAN ---
daftar_merek = ['TOYOTA', 'DAIHATSU', 'HONDA', 'MITSUBISHI', 'SUZUKI']
hasil_prediksi_final = []
kolom_fitur_final = [col for col in df_latih_baru.columns if col not in daftar_merek + ['waktu']]

for merek in daftar_merek:
    print(f"\n--- Memproses Merek: {merek} ---")

    y_latih = df_latih_baru[merek]
    X_latih = df_latih_baru[kolom_fitur_final]

    study = optuna.create_study(direction='minimize', study_name=f'lgbm_study_{merek}')
    study.optimize(lambda trial: objective(trial, X_latih, y_latih), n_trials=70, n_jobs=-1)

    best_params = study.best_params
    best_score = study.best_value
    print(f"sMAPE terbaik ditemukan untuk {merek}: {best_score:.2f}%")
    print(f"Parameter terbaik: {best_params}")

    final_model = LGBMRegressor(**best_params, random_state=404, n_jobs=-1)
    final_model.fit(X_latih, y_latih)

    X_prediksi = df_prediksi_baru[kolom_fitur_final]
    prediksi_agustus = final_model.predict(X_prediksi)
    nilai_prediksi = int(prediksi_agustus[0])

    print(f"Prediksi Penjualan {merek} Agustus 2025: {nilai_prediksi} unit")

    hasil_prediksi_final.append({
        'Merek': merek,
        'sMAPE_Optuna': best_score,
        'Prediksi_Agustus2025': nilai_prediksi
    })

# --- RINGKASAN AKHIR ---
df_ringkasan = pd.DataFrame(hasil_prediksi_final)
print("\n=================== RINGKASAN AKHIR ===================")
print(df_ringkasan)

[I 2025-09-06 16:18:12,356] A new study created in memory with name: lgbm_study_TOYOTA



--- Memproses Merek: TOYOTA ---


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 306 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 307 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 308 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 309 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 310 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The

sMAPE terbaik ditemukan untuk TOYOTA: 29.53%
Parameter terbaik: {'n_estimators': 612, 'learning_rate': 0.09649374308910638, 'num_leaves': 38, 'max_depth': 8, 'subsample': 0.638957751025338, 'colsample_bytree': 0.6002319534432183, 'reg_alpha': 0.07100455320645715, 'reg_lambda': 0.16263044329461507}
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 10, number of used features: 0
[LightGBM] [Info] Start training from score 11657.987598
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split require

Streaming output truncated to the last 5000 lines.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 547 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 548 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 549 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 550 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 551 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501:

...

sMAPE terbaik ditemukan untuk HONDA: 29.30%
Parameter terbaik: {'n_estimators': 503, 'learning_rate': 0.08679241900412075, 'num_leaves': 29, 'max_depth': 5, 'subsample': 0.8619220493720916, 'colsample_bytree': 0.9547537917334687, 'reg_alpha': 0.019566194619214605, 'reg_lambda': 0.9516919599671083}
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 10, number of used features: 0
[LightGBM] [Info] Start training from score 3031.734302
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirem

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 50 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 51 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 52 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 53 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 54 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The repo

sMAPE terbaik ditemukan untuk MITSUBISHI: 13.56%
Parameter terbaik: {'n_estimators': 446, 'learning_rate': 0.12573395407118623, 'num_leaves': 11, 'max_depth': 9, 'subsample': 0.6662284724949156, 'colsample_bytree': 0.842123911356756, 'reg_alpha': 0.5492507778731391, 'reg_lambda': 0.947483398640849}
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 10, number of used features: 0
[LightGBM] [Info] Start training from score 6277.665479
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split require

Streaming output truncated to the last 5000 lines.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 825 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 279 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 826 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 280 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 827 is already reported.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/optuna/trial/_trial.py:501:

sMAPE terbaik ditemukan untuk SUZUKI: 14.87%
Parameter terbaik: {'n_estimators': 310, 'learning_rate': 0.07515385564159081, 'num_leaves': 24, 'max_depth': 7, 'subsample': 0.9178314364277171, 'colsample_bytree': 0.8616462079194624, 'reg_alpha': 0.23712762007082488, 'reg_lambda': 0.5185892367124464}
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 10, number of used features: 0
[LightGBM] [Info] Start training from score 3011.530005
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirem

In [118]:
def smape(y_true, y_pred):
    numerator = 2 * np.abs(y_pred - y_true)
    denominator = np.abs(y_true) + np.abs(y_pred)
    ratio = np.where(denominator == 0, 0, numerator / denominator)
    return np.mean(ratio) * 100

daftar_merek = ['TOYOTA', 'DAIHATSU', 'HONDA', 'MITSUBISHI', 'SUZUKI']

# Simpan hasil evaluasi & prediksi
ringkasan_hasil = []

kolom_fitur = [col for col in df_latih.columns if col not in daftar_merek + ['waktu']]

for merek in daftar_merek:
    print(f"\n--- Memproses Merek: {merek} ---")

    y_latih = df_latih[merek]
    X_latih = df_latih[kolom_fitur]

    models = {
        "LightGBM": LGBMRegressor(
            random_state=404,
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            reg_alpha=0.2,
            reg_lambda=0.2,
            colsample_bytree=0.9,
            subsample=0.9
        ),
        "XGBoost": XGBRegressor(
            random_state=404,
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=7,
            subsample=0.9,
            colsample_bytree=0.9,
            gamma=0.2
        ),
        "CatBoost": CatBoostRegressor(
            random_state=404,
            verbose=0,
            iterations=1000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=5
        )
    }

    # Evaluasi model
    print("--- Mengevaluasi Performa Model ---")
    tscv = TimeSeriesSplit(n_splits=3)
    oof_preds = np.zeros((len(y_latih), len(models)))
    model_scores = {}

    for i, (model_name, model) in enumerate(models.items()):
        scores = []
        for train_idx, test_idx in tscv.split(X_latih):
            X_train, X_test = X_latih.iloc[train_idx], X_latih.iloc[test_idx]
            y_train, y_test = y_latih.iloc[train_idx], y_latih.iloc[test_idx]

            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            oof_preds[test_idx, i] = preds

            scores.append(smape(y_test, preds))

        avg_score = np.mean(scores)
        model_scores[model_name] = avg_score
        print(f"Estimasi sMAPE ({model_name}) untuk {merek}: {avg_score:.2f}%")

    # Stacking dengan Ridge sebagai meta-model
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(oof_preds, y_latih)

    # Fit ulang semua base model di full data
    final_preds = []
    for model_name, model in models.items():
        model.fit(X_latih, y_latih)
        final_preds.append(model.predict(df_prediksi[kolom_fitur]))

    # Gabungkan prediksi base model
    final_preds = np.column_stack(final_preds)
    ensemble_pred = meta_model.predict(final_preds)[0]

    print(f"Prediksi Penjualan (Stacked Ensemble) {merek} Agustus 2025: {int(ensemble_pred)}")

    # Simpan hasil ke ringkasan
    ringkasan_hasil.append({
        "Merek": merek,
        "sMAPE_LightGBM": model_scores["LightGBM"],
        "sMAPE_XGBoost": model_scores["XGBoost"],
        "sMAPE_CatBoost": model_scores["CatBoost"],
        "Prediksi_Ensemble_Agustus2025": int(ensemble_pred)
    })

# Buat DataFrame ringkasan
df_ringkasan = pd.DataFrame(ringkasan_hasil)
print("\n=================== RINGKASAN AKHIR ===================")
print(df_ringkasan)


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W